# Train the Model 3 LoRA adapter (QLoRA on a Colab T4)

**Status: written from the TRL 1.18.2 / PEFT 0.22.0 / Transformers 5.24 documentation of 2026-11-24. It has NOT been executed** (the machine that wrote it has no GPU).
Run it top to bottom on a free T4 (Runtime > Change runtime type > T4 GPU) and read every output before trusting the next cell.

What the adapter teaches is **behaviour**: answer only from the numbered passages, cite them as `[n]`, say "I could not find this in the available university documents" when they do not contain the answer.
University facts stay in retrieval. The training data was built by `scripts/build_peft_dataset.py` with the exact prompt format the app serves, and passages that prove any evaluation answer were held out.

**Before you start:** upload `train.jsonl`, `val.jsonl` and `dataset_manifest.json` (from `data/peft/`) to the Colab file panel, and add a Colab secret named `HF_TOKEN` (a Hugging Face token with write access).
The adapter is pushed to a **private** repository; never commit weights to Git.

In [ ]:
# 1. Install the versions this notebook was written against.
#    EXPECTED: a pip warning that cudf-cu13 wants pyarrow<24. It is harmless (cuDF is Colab's GPU dataframe library; this notebook never uses it).
#    AFTER THIS CELL: Runtime > Restart session (NOT "Disconnect and delete runtime", which would erase your uploaded files), then continue from cell 2.
%pip -q install "trl==1.18.2" "peft==0.22.0" "transformers==5.24.0" "bitsandbytes==0.50.2" "accelerate==1.16.0" "datasets==5.2.0" "huggingface_hub==2.4.0"

In [ ]:
# 2. Settings: edit HF_ADAPTER_REPO, nothing else is needed.
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # Apache-2.0, ungated, non-thinking instruct model
HF_ADAPTER_REPO = "your-hf-username/cu-helpdesk-lora"   # a PRIVATE repo; created if missing
SEED = 7
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA = {"r": 16, "alpha": 32, "dropout": 0.05}
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MAX_LENGTH = 2048
OUT = "adapter"

In [ ]:
# 3. The GPU decides the precision. bfloat16 needs an Ampere-or-newer GPU (compute capability 8 or more: A100, L4, ...). A T4 (7.5) trains in float16.
#    Do NOT use torch.cuda.is_bf16_supported() for this: it answers yes for a T4 (which can only emulate bfloat16) and TRL's kernels then fail.
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
major, minor = torch.cuda.get_device_capability(0)
BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), f"| compute capability {major}.{minor}", "| bf16" if BF16 else "| fp16", "| torch", torch.__version__)

# Prove the 4-bit kernels work on this GPU and CUDA version BEFORE the 8 GB model download (a broken bitsandbytes would otherwise fail much later).
import bitsandbytes as bnb
layer = bnb.nn.Linear4bit(64, 64, quant_type="nf4", compute_dtype=COMPUTE_DTYPE).to("cuda")
layer(torch.randn(1, 64, dtype=COMPUTE_DTYPE, device="cuda"))
print("bitsandbytes", bnb.__version__, "- 4-bit works | CUDA", torch.version.cuda)

In [ ]:
# 4. Data. The training file must be exactly the one the dataset manifest describes (the same hash the builder computed).
import hashlib, json
from datasets import load_dataset

def dataset_sha256(path):
    h = hashlib.sha256()
    for line in open(path, encoding="utf-8"):
        if line.strip():
            h.update(json.dumps(json.loads(line)["messages"], ensure_ascii=False, sort_keys=True).encode()); h.update(b"\n")
    return h.hexdigest()

dataset_manifest = json.load(open("dataset_manifest.json", encoding="utf-8"))
TRAIN_SHA = dataset_sha256("train.jsonl")
assert TRAIN_SHA == dataset_manifest["train_sha256"], "train.jsonl is not the file the manifest describes: upload the matching pair"
data = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
for split in data:
    for row in data[split]:
        roles = [m["role"] for m in row["messages"]]
        assert roles[0] == "system" and roles[-1] == "assistant" and "user" in roles, roles
print({k: len(v) for k, v in data.items()}, "| prompt version:", dataset_manifest["prompt_version"], "| sha256", TRAIN_SHA[:16])
print(data["train"][0]["messages"][-1]["content"][:200])

In [ ]:
# 5. Pin the base model to an exact commit (a branch name can change under you), then load it in 4 bit (NF4, double quantisation).
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

BASE_REVISION = HfApi().model_info(BASE_MODEL).sha
print("base model commit:", BASE_REVISION)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION, quantization_config=bnb, device_map="auto", dtype=COMPUTE_DTYPE)
model = prepare_model_for_kbit_training(model)   # the PEFT quantisation guide calls this before adding the adapter
peft_config = LoraConfig(r=LORA["r"], lora_alpha=LORA["alpha"], lora_dropout=LORA["dropout"], target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM")

In [ ]:
# 6. Train. assistant_only_loss=True trains only on the assistant's answers (TRL patches the Qwen3 chat template for this); if the
#    template cannot support it the trainer raises, and the fallback trains on every token and says so in the manifest.
if not BF16:
    # TRL 1.18 always computes the loss with a fused kernel that asks the GPU for float32-output matrix products on half-precision inputs. Older GPUs
    # (a T4) may not support that; this switches the kernel to its own plain float32 fallback (slightly slower, works everywhere).
    import trl.kernels.chunked_logprob as _chunked_logprob
    _chunked_logprob._MM_OUT_DTYPE = False
from trl import SFTConfig, SFTTrainer

def make_trainer(assistant_only):
    cfg = SFTConfig(
        output_dir=OUT, max_length=MAX_LENGTH, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_steps=0.03,  # a float below 1 means this share of all steps (the current TrainingArguments has no warmup_ratio)
        per_device_train_batch_size=2, gradient_accumulation_steps=8, per_device_eval_batch_size=2,
        fp16=not BF16, bf16=BF16, logging_steps=10, eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        assistant_only_loss=assistant_only, report_to="none", seed=SEED,
    )
    return SFTTrainer(model=model, args=cfg, train_dataset=data["train"], eval_dataset=data["validation"], processing_class=tokenizer, peft_config=peft_config)

ASSISTANT_ONLY = True
try:
    trainer = make_trainer(True)
except (ValueError, RuntimeError) as exc:
    print("assistant_only_loss is not supported by this chat template, training on all tokens instead:", exc)
    ASSISTANT_ONLY = False
    trainer = make_trainer(False)
trainer.train()
metrics = trainer.evaluate()
print(metrics)

In [ ]:
# 7. Save the adapter (adapter_config.json and adapter_model.safetensors) and look at one held-out example.
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)

example = data["validation"][0]["messages"]
prompt = tokenizer.apply_chat_template(example[:-1], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
out = trainer.model.generate(**inputs, max_new_tokens=120, do_sample=False)
print("MODEL:    ", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print("REFERENCE:", example[-1]["content"])

In [ ]:
# 8. The manifest: what the adapter is, what it was trained on and with. scripts/check_model3_endpoint.py --manifest validates it.
import datetime, platform, importlib.metadata as md_
manifest = {
    "base_model_id": BASE_MODEL, "base_model_revision": BASE_REVISION, "dataset_sha256": TRAIN_SHA,
    "lora": {**LORA, "target_modules": TARGET_MODULES}, "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "seed": SEED, "max_length": MAX_LENGTH,
    "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "prompt_version": dataset_manifest["prompt_version"],
    "assistant_only_loss": ASSISTANT_ONLY, "eval_loss": metrics.get("eval_loss"),
    "examples": dataset_manifest["examples"], "golden_set_sha256": dataset_manifest["golden_set"]["sha256"],
    "libraries": {p: md_.version(p) for p in ("trl", "peft", "transformers", "bitsandbytes", "accelerate", "datasets", "torch")},
    "hardware": torch.cuda.get_device_name(0), "python": platform.python_version(),
}
json.dump(manifest, open(f"{OUT}/adapter_manifest.json", "w", encoding="utf-8"), indent=2)
print(json.dumps(manifest, indent=2))

In [ ]:
# 9. Push to a PRIVATE Hugging Face repo and print the lines for .env. The commit hash pins exactly this adapter.
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
api.create_repo(HF_ADAPTER_REPO, private=True, exist_ok=True)
commit = api.upload_folder(folder_path=OUT, repo_id=HF_ADAPTER_REPO, commit_message=f"LoRA adapter, dataset {TRAIN_SHA[:12]}")
print("commit:", commit.oid)
print(f'''
MODEL3_MODEL=cu-helpdesk-lora
MODEL3_BASE_MODEL_ID={BASE_MODEL}
MODEL3_ADAPTER_ID={HF_ADAPTER_REPO}@{commit.oid}
MODEL3_ABLATION_MODEL={BASE_MODEL}
# also set these when the endpoint is deployed (deploy/modal_vllm.py):
# MODEL3_BASE_URL=https://<your-endpoint>/v1   MODEL3_API_KEY=<the key you gave vLLM>
''')